In [1]:
import os
import sys

In [2]:
sys.path.append(os.path.abspath(".."))

In [3]:
from datalemur_utils import extract_iframe_from_url

In [4]:
url = "https://datalemur.com/questions/sql-page-with-no-likes"

In [5]:
extract_iframe_from_url(url)

In [6]:
from pyspark.sql import SparkSession

In [7]:
spark = SparkSession.builder.getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/02 17:45:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [8]:
pages = spark.read\
.option("header","true")\
.option("inferSchema", "false")\
.schema("page_id int, page_name string")\
.format("csv")\
.load("pages.txt")

In [9]:
pages.show(truncate=False)

[Stage 0:>                                                          (0 + 1) / 1]

+-------+----------------------+
|page_id|page_name             |
+-------+----------------------+
|20001  |SQL Solutions         |
|20045  |Brain Exercises       |
|20701  |Tips for Data Analysts|
+-------+----------------------+



In [10]:
pages.createOrReplaceTempView("pages")

In [11]:
page_likes = spark.read\
.option("header","true")\
.option("delimiter", "\t")\
.option("inferSchema", "false")\
.schema("user_id int, page_id int, liked_date string")\
.format("csv")\
.load("page_likes.txt")

In [12]:
page_likes.show(truncate=False)

+-------+-------+-------------------+
|user_id|page_id|liked_date         |
+-------+-------+-------------------+
|111    |20001  |04/08/2022 00:00:00|
|121    |20045  |03/12/2022 00:00:00|
|156    |20001  |07/25/2022 00:00:00|
+-------+-------+-------------------+



In [13]:
page_likes.createOrReplaceTempView("page_likes")

In [14]:
spark.sql(
    """
    select p.page_id as page_id from pages p
    left join page_likes pl on p.page_id=pl.page_id
    where pl.page_id is null
    """
).show(truncate=False)

+-------+
|page_id|
+-------+
|20701  |
+-------+



In [15]:
pages.join(page_likes, on = pages.page_id==page_likes.page_id, how = "left")\
.filter(page_likes.page_id.isNull())\
.select(pages.page_id)\
.show(truncate=False)

+-------+
|page_id|
+-------+
|20701  |
+-------+



In [16]:
spark.stop()